In [1]:
# =====================================================
# MÓDULO 4
# Evaluación Nutricional
# Dataset: Food Nutrition Dataset
# Algoritmos:
#   - Decision Tree
#   - Random Forest
#
# Fix aplicado (revisión backend, 2026-07-25):
#   El Nutrition_Score se calculaba con calories/protein/fat, y esas
#   MISMAS columnas se dejaban dentro de X -> data leakage: el
#   modelo no predecia nada, solo memorizaba el umbral que ya
#   conociamos (por eso daba ~93% "accuracy" con clases de 1-3
#   muestras). Fix: calories/protein/fat y food_name (identificador,
#   no generaliza) se sacan de X. Quedan category/carbs/iron/vitamin_c
#   como señal real, no derivada del target.
#
#   Aviso importante para el equipo: este dataset es POR ALIMENTO
#   (206 filas), no por usuario/dia como pide la Fase 4 del
#   documento (DailyCalories, Sugar, FruitsServings, etc.). Aunque
#   el leakage ya esta arreglado, este modelo clasifica alimentos
#   individuales, no el consumo diario de un usuario -> el backend
#   NO puede alimentarlo con lo que el usuario llena en el
#   formulario. Recomendacion: para el score nutricional en vivo,
#   calcular el Nutrition Score de la app por FORMULA sobre los
#   totales diarios reportados (igual que un cuestionario tipo
#   FINDRISC), y dejar este modelo como pieza exploratoria del curso
#   hasta que exista un dataset a nivel usuario/dia.
# =====================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

df = pd.read_csv("../data/Food_Nutrition_Dataset.csv")
df = df.drop_duplicates()

print("Primeras filas")
print(df.head())
print("\nValores nulos")
print(df.isnull().sum())

for col in df.select_dtypes(include=["float64", "int64"]).columns:
    df[col] = df[col].fillna(df[col].median())

def nutrition_score(row):
    if row["calories"] <= 150 and row["protein"] >= 5 and row["fat"] <= 5:
        return "Excelente"
    elif row["calories"] <= 300 and row["protein"] >= 3 and row["fat"] <= 10:
        return "Buena"
    elif row["calories"] <= 500:
        return "Regular"
    else:
        return "Deficiente"

df["Nutrition_Score"] = df.apply(nutrition_score, axis=1)

print("\nDistribución del Nutrition Score")
print(df["Nutrition_Score"].value_counts())

# Se excluyen food_name (identificador) y calories/protein/fat
# (definen el target) para evitar leakage.
X = df.drop(columns=["Nutrition_Score", "food_name", "calories", "protein", "fat"])
y = df["Nutrition_Score"]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# =====================================================
# MODELO 1 - DECISION TREE
# =====================================================

modelo_dt = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("modelo", DecisionTreeClassifier(random_state=42, class_weight="balanced"))
])
modelo_dt.fit(X_train, y_train)
pred_dt = modelo_dt.predict(X_test)

print("\n==============================")
print("DECISION TREE (sin leakage)")
print("==============================")
print("Accuracy :", round(accuracy_score(y_test, pred_dt), 4))
print("Precision:", round(precision_score(y_test, pred_dt, average="weighted", zero_division=1), 4))
print("Recall   :", round(recall_score(y_test, pred_dt, average="weighted", zero_division=1), 4))
print("F1 Score :", round(f1_score(y_test, pred_dt, average="weighted", zero_division=1), 4))
print("\nMatriz de Confusión")
print(confusion_matrix(y_test, pred_dt))
print("\nClassification Report")
print(classification_report(y_test, pred_dt, zero_division=1))

# =====================================================
# MODELO 2 - RANDOM FOREST
# =====================================================

modelo_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("modelo", RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight="balanced"
    ))
])
modelo_rf.fit(X_train, y_train)
pred_rf = modelo_rf.predict(X_test)

print("\n==============================")
print("RANDOM FOREST (sin leakage)")
print("==============================")
print("Accuracy :", round(accuracy_score(y_test, pred_rf), 4))
print("Precision:", round(precision_score(y_test, pred_rf, average="weighted", zero_division=1), 4))
print("Recall   :", round(recall_score(y_test, pred_rf, average="weighted", zero_division=1), 4))
print("F1 Score :", round(f1_score(y_test, pred_rf, average="weighted", zero_division=1), 4))
print("\nMatriz de Confusión")
print(confusion_matrix(y_test, pred_rf))
print("\nClassification Report")
print(classification_report(y_test, pred_rf, zero_division=1))

# =====================================================
# COMPARACIÓN FINAL Y SELECCIÓN DEL GANADOR
# =====================================================

resultados = pd.DataFrame({
    "Modelo": ["Decision Tree", "Random Forest"],
    "Accuracy": [accuracy_score(y_test, pred_dt), accuracy_score(y_test, pred_rf)],
    "Precision": [
        precision_score(y_test, pred_dt, average="weighted", zero_division=1),
        precision_score(y_test, pred_rf, average="weighted", zero_division=1),
    ],
    "Recall": [
        recall_score(y_test, pred_dt, average="weighted", zero_division=1),
        recall_score(y_test, pred_rf, average="weighted", zero_division=1),
    ],
    "F1": [
        f1_score(y_test, pred_dt, average="weighted", zero_division=1),
        f1_score(y_test, pred_rf, average="weighted", zero_division=1),
    ],
})

print("\n==============================")
print("COMPARACIÓN DE MODELOS")
print("==============================")
print(resultados)

candidatos = {"Decision Tree": modelo_dt, "Random Forest": modelo_rf}
ganador_nombre = resultados.sort_values("Recall", ascending=False).iloc[0]["Modelo"]
ganador_pipeline = candidatos[ganador_nombre]

print(f"\nModelo ganador por Recall: {ganador_nombre}")

joblib.dump(ganador_pipeline, "../models_artifacts/modelo4_nutricion.joblib")
print("Guardado en ../models_artifacts/modelo4_nutricion.joblib")
print("Columnas esperadas por el modelo:", list(X.columns))
print("\nNOTA: este modelo opera sobre columnas por-alimento (category, carbs, iron, vitamin_c).")
print("No coincide con el formulario de la app (nutricion diaria del usuario). Ver nota arriba.")


Primeras filas
        food_name        category  calories  protein  carbs   fat  iron  \
0  Apple, candied          Apples     134.0     1.34  29.61  2.15  0.12   
1      Apple, raw          Apples      61.0     0.17  14.80  0.15  0.03   
2    Apple, dried    Dried fruits     243.0     0.93  65.89  0.32  1.40   
3    Crisp, apple  Cakes and pies     215.0     2.81  30.18  9.59  1.00   
4    Apple, baked          Apples     113.0     0.32  22.70  3.08  0.19   

   vitamin_c  
0        3.6  
1        4.6  
2        3.9  
3        0.6  
4        3.9  

Valores nulos
food_name    0
category     0
calories     0
protein      0
carbs        0
fat          0
iron         2
vitamin_c    3
dtype: int64

Distribución del Nutrition Score
Nutrition_Score
Regular       164
Deficiente     22
Buena          14
Excelente       5
Name: count, dtype: int64

DECISION TREE (sin leakage)
Accuracy : 0.6829
Precision: 0.7073
Recall   : 0.6829
F1 Score : 0.6829

Matriz de Confusión
[[ 0  1  0  2]
 [ 0  1  0 

/tmp/ipykernel_114343/2602808637.py:87: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns



RANDOM FOREST (sin leakage)
Accuracy : 0.6829
Precision: 0.7498
Recall   : 0.6829
F1 Score : 0.6998

Matriz de Confusión
[[ 0  1  0  2]
 [ 1  2  0  1]
 [ 0  0  0  1]
 [ 3  4  0 26]]

Classification Report
              precision    recall  f1-score   support

       Buena       0.00      0.00      0.00         3
  Deficiente       0.29      0.50      0.36         4
   Excelente       1.00      0.00      0.00         1
     Regular       0.87      0.79      0.83        33

    accuracy                           0.68        41
   macro avg       0.54      0.32      0.30        41
weighted avg       0.75      0.68      0.70        41


COMPARACIÓN DE MODELOS
          Modelo  Accuracy  Precision    Recall        F1
0  Decision Tree  0.682927   0.707317  0.682927  0.682927
1  Random Forest  0.682927   0.749826  0.682927  0.699821

Modelo ganador por Recall: Decision Tree
Guardado en ../models_artifacts/modelo4_nutricion.joblib
Columnas esperadas por el modelo: ['category', 'carbs', 'iron'